[<img src="https://colab.research.google.com/assets/colab-badge.svg" align="left" alt="Open In Colab"/>](https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)

<br><br>


# Apigee AI Gateway — MCP Gateway Governance & Policies (AFT)

**Repository Location:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb`](https://github.com/luiscuellarh/apigee-unified-ai-gateway/blob/main/03_apigee_mcp_gateway.ipynb)  
**Execution Constraint:** This notebook is version-controlled in GitHub and designed to run cleanly when opened directly from that repository path in Google Colab (no local-only paths, no machine-specific assumptions, and no state carried between sessions).

**Framework & Toolkit:**
- **Toolkit:** **AFT (`aft` — Apigee Foundation Toolkit / Apigee Feature Templater)** (`https://github.com/apigee/apigee-templater`)
- **Repository:** [`https://github.com/luiscuellarh/apigee-unified-ai-gateway`](https://github.com/luiscuellarh/apigee-unified-ai-gateway)
- **Assumptions:** Apigee X is already provisioned with `APIGEE_ENV` attached to an environment group (`ENV_GROUP`), and target MCP backend services are reachable.

---

## Step 0 — Preflight (Single-Run Blocking Gate)
Run **Step 0** once to execute all four preflight stages (`0.1`–`0.4`) in a single execution:
- **0.1** Inputs & Global Configuration
- **0.2** Verify Apigee Topology, APIs & IAM
- **0.3** Verify Service Account Roles, Permissions & `<GoogleAccessToken>` Bindings + Live Gemini
- **0.4** Install AFT CLI & Preflight Summary Gate


In [ ]:
#@title 0.1–0.4 Step 0 Preflight (Inputs, Apigee Topology & IAM, Service Account & Live Gemini, AFT CLI & Summary Gate)
# Runs 0.1, 0.2, 0.3, and 0.4 sequentially in a single execution.
import json, os, pathlib, re, shutil, subprocess, time, urllib.error, urllib.request
import requests

# ============================================================================
# 0.1 Inputs & Global Configuration
# ============================================================================
print("▶ [0.1] Inputs & Global Configuration...")
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown ### Project & Apigee
PROJECT_ID = "" #@param {type:"string"}
APIGEE_ORG = "" #@param {type:"string"}
APIGEE_ENV = "dev" #@param {type:"string"}
ENV_GROUP = "dev-group" #@param {type:"string"}
REGION = "europe-west1" #@param {type:"string"}
HOSTNAME = "" #@param {type:"string"}

#@markdown ### Service Account
SERVICE_ACCOUNT = "" #@param {type:"string"}

#@markdown ### Gemini
GEMINI_MODEL = "gemini-2.5-flash" #@param {type:"string"}

#@markdown ### Behaviour Flags
ENABLE_APIS = False #@param {type:"boolean"}

PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT = (
    v.strip() for v in (PROJECT_ID, APIGEE_ENV, ENV_GROUP, REGION, HOSTNAME, SERVICE_ACCOUNT)
)
APIGEE_ORG = APIGEE_ORG.strip() or PROJECT_ID
APIGEE_ENVIRONMENT = APIGEE_ENV
GEMINI_MODEL = GEMINI_MODEL.strip().removeprefix("google/")

missing = [k for k, v in {"PROJECT_ID": PROJECT_ID, "APIGEE_ENV": APIGEE_ENV, "REGION": REGION, "GEMINI_MODEL": GEMINI_MODEL}.items() if not v]
if missing:
    raise SystemExit(f"❌ HALT: Missing required parameter(s): {', '.join(missing)}.")

# If SERVICE_ACCOUNT is provided, do not ask for a name and do not create a new one
SA_PROVIDED = bool(SERVICE_ACCOUNT)
if SERVICE_ACCOUNT and "@" not in SERVICE_ACCOUNT:
    SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
if SERVICE_ACCOUNT and not re.fullmatch(r"[a-zA-Z0-9._-]+@[a-zA-Z0-9.-]+\.gserviceaccount\.com", SERVICE_ACCOUNT):
    raise SystemExit(f"❌ HALT: Invalid SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' (expected *.gserviceaccount.com).")

os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "APIGEE_ORG": APIGEE_ORG,
    "APIGEE_ENV": APIGEE_ENV,
    "APIGEE_ENVIRONMENT": APIGEE_ENVIRONMENT,
    "ENV_GROUP": ENV_GROUP,
    "REGION": REGION,
    "HOSTNAME": HOSTNAME,
    "SERVICE_ACCOUNT": SERVICE_ACCOUNT,
    "GEMINI_MODEL": GEMINI_MODEL,
})
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True, capture_output=True)

PREFLIGHT = {"inputs": True, "apigee": False, "service_account": False, "gemini": False, "aft": False, "repo": False}
sa_mode = f"{SERVICE_ACCOUNT} (provided — verify only, no creation)" if SA_PROVIDED else f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com (auto-provision if missing)"
print(f"✅ [0.1] Inputs ready | project={PROJECT_ID} | org={APIGEE_ORG} | env={APIGEE_ENV} ({ENV_GROUP}) | region={REGION} | sa={sa_mode}")

# ============================================================================
# 0.2 Verify Apigee Topology, APIs & IAM
# ============================================================================
print("\n▶ [0.2] Verify Apigee Topology, APIs & IAM...")
def get_auth_token() -> str:
    return subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()

def gcp_request(method: str, url: str, json_body=None, headers=None):
    req_headers = {"Authorization": f"Bearer {get_auth_token()}", "Content-Type": "application/json"}
    if headers:
        req_headers.update(headers)
    try:
        resp = requests.request(method, url, json=json_body, headers=req_headers, timeout=30)
        return resp.status_code, (resp.json() if resp.text else {})
    except ValueError:
        return resp.status_code, {"raw": resp.text}
    except Exception as exc:
        return 0, {"error": str(exc), "url": url}

def wait_for_proxy_ready(proxy_name: str, env_name: str = None, timeout_sec: int = 180) -> str:
    env_target = env_name or APIGEE_ENV
    list_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/deployments"
    deadline = time.time() + timeout_sec
    last_rev, last_state = None, "NOT_DEPLOYED"
    print(f"⏳ Waiting for proxy '{proxy_name}' to reach READY state in env '{env_target}'...")
    while time.time() < deadline:
        s_list, d_list = gcp_request("GET", list_url)
        deployments = d_list.get("deployments", []) if s_list == 200 else []
        if deployments:
            latest = sorted(deployments, key=lambda d: int(d.get("revision", 0)))[-1]
            last_rev = latest.get("revision")
            rev_url = f"{APIGEE_BASE}/environments/{env_target}/apis/{proxy_name}/revisions/{last_rev}/deployments"
            s_rev, d_rev = gcp_request("GET", rev_url)
            last_state = d_rev.get("state", latest.get("state", "UNKNOWN")) if s_rev == 200 else "UNKNOWN"
            if last_state == "READY":
                print(f"✅ Proxy '{proxy_name}' (revision {last_rev}) is READY in env '{env_target}'.")
                return str(last_rev)
            print(f"   • proxy={proxy_name} | rev={last_rev} | state={last_state} — waiting 5s...")
        else:
            print(f"   • proxy={proxy_name} | state=WAITING_FOR_DEPLOYMENT — waiting 5s...")
        time.sleep(5)
    raise SystemExit(
        f"❌ HALT: Timed out waiting for proxy '{proxy_name}' to become READY in env '{env_target}' "
        f"(last revision={last_rev}, state={last_state})."
    )

ACTIVE_IDENTITY = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()
if not ACTIVE_IDENTITY:
    _, t_info = gcp_request("GET", f"https://oauth2.googleapis.com/tokeninfo?access_token={get_auth_token()}")
    ACTIVE_IDENTITY = t_info.get("email", "")
if not ACTIVE_IDENTITY:
    raise SystemExit("❌ HALT: Could not resolve active caller identity.")
caller_member_prefix = "serviceAccount" if ACTIVE_IDENTITY.endswith(".gserviceaccount.com") else "user"
print(f"✅ Active identity: {ACTIVE_IDENTITY}")

# 1. Verify required GCP APIs
REQUIRED_SERVICES = ["apigee.googleapis.com", "cloudresourcemanager.googleapis.com", "iamcredentials.googleapis.com", "aiplatform.googleapis.com"]
disabled_svcs = [
    svc for svc in REQUIRED_SERVICES
    if gcp_request("GET", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services/{svc}")[1].get("state") != "ENABLED"
]
if disabled_svcs:
    if ENABLE_APIS or disabled_svcs == ["iamcredentials.googleapis.com"]:
        gcp_request("POST", f"https://serviceusage.googleapis.com/v1/projects/{PROJECT_ID}/services:batchEnable", {"serviceIds": disabled_svcs})
        time.sleep(3)
    else:
        raise SystemExit(f"❌ HALT: Disabled API(s): {', '.join(disabled_svcs)}. Set ENABLE_APIS=True in Step 0 and re-run.")
print(f"✅ Required APIs ENABLED: {', '.join(REQUIRED_SERVICES)}")

# 2. Validate Apigee Organization, Environment, and Environment Group Hostname
APIGEE_BASE = f"https://apigee.googleapis.com/v1/organizations/{APIGEE_ORG}"
status, org_data = gcp_request("GET", APIGEE_BASE)
if status != 200 or org_data.get("state") != "ACTIVE":
    raise SystemExit(f"❌ HALT: Apigee org '{APIGEE_ORG}' not ACTIVE (HTTP {status}): {org_data}")

status, env_data = gcp_request("GET", f"{APIGEE_BASE}/environments/{APIGEE_ENV}")
if status != 200:
    raise SystemExit(f"❌ HALT: Apigee env '{APIGEE_ENV}' not found in org '{APIGEE_ORG}' (HTTP {status}): {env_data}")

_, eg_list = gcp_request("GET", f"{APIGEE_BASE}/envgroups")
matched_group, matched_hosts = None, []
for eg in eg_list.get("environmentGroups", []):
    eg_name = eg["name"]
    if ENV_GROUP and eg_name != ENV_GROUP:
        continue
    _, att_data = gcp_request("GET", f"{APIGEE_BASE}/envgroups/{eg_name}/attachments")
    if APIGEE_ENV in [a.get("environment") for a in att_data.get("environmentGroupAttachments", [])]:
        matched_group, matched_hosts = eg_name, eg.get("hostnames", [])
        break

if not matched_group or (not HOSTNAME and not matched_hosts):
    raise SystemExit(f"❌ HALT: Env '{APIGEE_ENV}' is not attached to a valid envgroup with a hostname in '{APIGEE_ORG}'.")
ENV_GROUP, HOSTNAME = matched_group, (HOSTNAME or matched_hosts[0])
os.environ.update({"ENV_GROUP": ENV_GROUP, "HOSTNAME": HOSTNAME})
print(f"✅ Apigee Topology: org={APIGEE_ORG} | env={APIGEE_ENV} | envgroup={ENV_GROUP} | host={HOSTNAME}")

# 3. Verify Effective Apigee IAM Permissions
REQUIRED_APIGEE_PERMS = [
    "apigee.proxies.create", "apigee.proxies.get", "apigee.proxies.update", "apigee.proxies.delete",
    "apigee.proxyrevisions.get", "apigee.proxyrevisions.update", "apigee.proxyrevisions.delete",
    "apigee.proxyrevisions.deploy", "apigee.proxyrevisions.undeploy",
    "apigee.deployments.create", "apigee.deployments.get", "apigee.deployments.update", "apigee.deployments.delete",
    "apigee.sharedflows.create", "apigee.sharedflows.get", "apigee.sharedflows.delete",
    "apigee.sharedflowrevisions.update", "apigee.sharedflowrevisions.deploy", "apigee.sharedflowrevisions.undeploy",
    "apigee.keyvaluemaps.create", "apigee.keyvaluemaps.get", "apigee.keyvaluemaps.update", "apigee.keyvaluemaps.delete",
    "apigee.keyvaluemapentries.create", "apigee.keyvaluemapentries.get", "apigee.keyvaluemapentries.update", "apigee.keyvaluemapentries.delete",
    "apigee.targetservers.create", "apigee.targetservers.get", "apigee.targetservers.update", "apigee.targetservers.delete",
]
iam_url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}:testIamPermissions"
_, iam_resp = gcp_request("POST", iam_url, {"permissions": REQUIRED_APIGEE_PERMS})
missing_perms = [p for p in REQUIRED_APIGEE_PERMS if p not in set(iam_resp.get("permissions", []))]
if missing_perms:
    raise SystemExit(f"❌ HALT: Missing Apigee IAM permissions: {', '.join(missing_perms)}")
PREFLIGHT["apigee"] = True
print(f"✅ [0.2] Verified all {len(REQUIRED_APIGEE_PERMS)} Apigee IAM permissions.")

# ============================================================================
# 0.3 Verify Service Account Roles, Permissions & <GoogleAccessToken> Bindings + Live Gemini
# ============================================================================
print("\n▶ [0.3] Verify Service Account Roles, Permissions & Bindings + Live Gemini...")
REQUIRED_SA_ROLES = ["roles/aiplatform.user", "roles/iam.serviceAccountTokenCreator"]

def ensure_policy_bindings(get_url: str, set_url: str, required_pairs: list, label: str):
    status, policy = gcp_request("POST", get_url, {})
    if status != 200:
        raise SystemExit(f"❌ HALT: Failed to read {label} IAM policy (HTTP {status}): {policy}")
    bindings = policy.setdefault("bindings", [])
    changed = False
    for member, role in required_pairs:
        entry = next((b for b in bindings if b.get("role") == role and not b.get("condition")), None)
        if entry and member in entry.get("members", []):
            print(f"  ✔ {label} verified: {role} -> {member}")
        else:
            print(f"  ⏳ Binding {label}: {role} -> {member}")
            if entry:
                entry.setdefault("members", []).append(member)
            else:
                bindings.append({"role": role, "members": [member]})
            changed = True
    if changed:
        s_code, s_resp = gcp_request("POST", set_url, {"policy": policy})
        if s_code != 200:
            raise SystemExit(f"❌ HALT: Failed to update {label} IAM policy (HTTP {s_code}): {s_resp}")
        print(f"  ✔ {label} bindings updated.")

# 1. Resolve Project Number & Apigee Service Agent
_, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{APIGEE_ORG}")
if not proj_data.get("projectNumber"):
    _, proj_data = gcp_request("GET", f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}")
if not proj_data.get("projectNumber"):
    raise SystemExit(f"❌ HALT: Unable to resolve projectNumber for '{PROJECT_ID}': {proj_data}")

PROJECT_NUMBER = str(proj_data["projectNumber"])
APIGEE_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-apigee.iam.gserviceaccount.com"

# 2. Verify Provided Service Account (or Auto-Provision Default Only When Not Provided)
target_sa = SERVICE_ACCOUNT or f"apigee-ai-gateway-sa@{PROJECT_ID}.iam.gserviceaccount.com"
sa_status, sa_data = gcp_request("GET", f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{target_sa}")

if SA_PROVIDED:
    if sa_status != 200 or sa_data.get("disabled", False):
        raise SystemExit(f"❌ HALT: Provided SERVICE_ACCOUNT '{SERVICE_ACCOUNT}' is missing or disabled (HTTP {sa_status}).")
    SA_STATUS = "VERIFIED_EXISTING"
    print(f"✅ Verified provided SERVICE_ACCOUNT: {SERVICE_ACCOUNT}")
elif sa_status == 200:
    SERVICE_ACCOUNT, SA_STATUS = target_sa, "REUSED_DEFAULT"
elif sa_status == 404:
    c_status, c_data = gcp_request(
        "POST",
        f"https://iam.googleapis.com/v1/projects/{PROJECT_ID}/serviceAccounts",
        {"accountId": "apigee-ai-gateway-sa", "serviceAccount": {"displayName": "Apigee AI Gateway Service Account"}},
    )
    if c_status not in (200, 201):
        raise SystemExit(f"❌ HALT: Failed to create default service account '{target_sa}' (HTTP {c_status}): {c_data}")
    SERVICE_ACCOUNT, SA_STATUS = c_data.get("email", target_sa), "CREATED_DEFAULT"
    time.sleep(3)
else:
    raise SystemExit(f"❌ HALT: Unable to inspect service account '{target_sa}' (HTTP {sa_status}): {sa_data}")

# 3. Verify & Ensure Project Roles (Gemini + Token Creator) and SA Resource Bindings (<GoogleAccessToken>)
proj_iam_base = f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}"
ensure_policy_bindings(
    f"{proj_iam_base}:getIamPolicy",
    f"{proj_iam_base}:setIamPolicy",
    [(f"serviceAccount:{SERVICE_ACCOUNT}", r) for r in REQUIRED_SA_ROLES],
    "Project role",
)

sa_iam_base = f"https://iam.googleapis.com/v1/projects/-/serviceAccounts/{SERVICE_ACCOUNT}"
ensure_policy_bindings(
    f"{sa_iam_base}:getIamPolicy",
    f"{sa_iam_base}:setIamPolicy",
    [
        (f"serviceAccount:{APIGEE_SERVICE_AGENT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountTokenCreator"),
        (f"serviceAccount:{SERVICE_ACCOUNT}", "roles/iam.serviceAccountUser"),
        (f"{caller_member_prefix}:{ACTIVE_IDENTITY}", "roles/iam.serviceAccountUser"),
    ],
    "SA <GoogleAccessToken> binding",
)

os.environ["SERVICE_ACCOUNT"] = SERVICE_ACCOUNT
PREFLIGHT["service_account"] = True
print(f"✅ Service Account ready for Gemini & <GoogleAccessToken>: {SERVICE_ACCOUNT} ({SA_STATUS})")

# 4. Verify Live Gemini Connectivity
payload = {"contents": [{"role": "user", "parts": [{"text": "Reply with READY."}]}], "generationConfig": {"maxOutputTokens": 8, "temperature": 0.0}}
def invoke_gemini(loc: str):
    host = "aiplatform.googleapis.com" if loc == "global" else f"{loc}-aiplatform.googleapis.com"
    url = f"https://{host}/v1/projects/{PROJECT_ID}/locations/{loc}/publishers/google/models/{GEMINI_MODEL}:generateContent"
    s, d = gcp_request("POST", url, payload, headers={"x-goog-user-project": PROJECT_ID})
    return gcp_request("POST", url, payload) if s == 403 else (s, d)

VERTEX_LOCATION, gen_status, gen_data = REGION, 0, {}
for _ in range(12):
    gen_status, gen_data = invoke_gemini(REGION)
    if gen_status == 200:
        VERTEX_LOCATION = REGION
        break
    for fb in [loc for loc in ("global", "us-central1") if loc != REGION]:
        fb_s, fb_d = invoke_gemini(fb)
        if fb_s == 200:
            gen_status, gen_data, VERTEX_LOCATION = fb_s, fb_d, fb
            break
    if gen_status == 200 or gen_status not in (403, 404):
        break
    time.sleep(5)

if gen_status != 200:
    raise SystemExit(f"❌ HALT: Live Gemini call failed (HTTP {gen_status}): {gen_data}")

os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION
PREFLIGHT["gemini"] = True
print(f"✅ [0.3] Live Gemini verified: model={GEMINI_MODEL} | location={VERTEX_LOCATION}")

# ============================================================================
# 0.4 Install AFT CLI & Preflight Summary Gate
# ============================================================================
print("\n▶ [0.4] Install AFT CLI & Preflight Summary Gate...")
os.environ.pop("AFT_REPOSITORY", None)
for extra_bin in [str(pathlib.Path.home() / ".aft" / "bin"), str(pathlib.Path.home() / ".local" / "bin")]:
    if extra_bin not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra_bin}:{os.environ['PATH']}"

if not shutil.which("aft"):
    subprocess.run("curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh", shell=True, check=True)

aft_bin = shutil.which("aft")
if not aft_bin:
    raise SystemExit("❌ HALT: 'aft' binary not found on PATH after installation.")
aft_ver = subprocess.run([aft_bin, "--version"], capture_output=True, text=True, check=True).stdout.strip()
PREFLIGHT["aft"] = True
print(f"✅ AFT CLI ready: {aft_ver} ({aft_bin})")

# 2. Clone GitHub Repository (local templates & features for AFT)
REPO_URL = "https://github.com/luiscuellarh/apigee-unified-ai-gateway.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path.cwd() / "apigee-unified-ai-gateway"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True, capture_output=True, text=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True, capture_output=True, text=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True, capture_output=True, text=True)

if not (REPO_DIR / "templates").is_dir() or not (REPO_DIR / "features").is_dir():
    raise SystemExit(f"❌ HALT: Cloned repository at '{REPO_DIR}' is missing 'templates/' or 'features/' directories.")

WORKSPACE_DIR = REPO_DIR
os.environ.update({"REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": str(REPO_DIR), "WORKSPACE_DIR": str(WORKSPACE_DIR)})
PREFLIGHT["repo"] = True
num_templates = len(list((REPO_DIR / "templates").glob("*.yaml")))
num_features = len(list((REPO_DIR / "features").glob("*.yaml")))
print(f"✅ Cloned GitHub repo ({REPO_BRANCH}): {REPO_DIR} ({num_templates} templates, {num_features} features)")

# 3. Preflight Summary Gate
failed_gates = [k for k, v in PREFLIGHT.items() if not v]
if failed_gates:
    raise SystemExit(f"❌ HALT: Preflight incomplete. Failed gate(s): {', '.join(failed_gates)}")

print("=" * 72)
for label, val in [
    ("GCP Project / No.",  f"{PROJECT_ID} ({PROJECT_NUMBER})"),
    ("Apigee Org / Env",   f"{APIGEE_ORG} / {APIGEE_ENV} ({ENV_GROUP})"),
    ("Gateway Hostname",   HOSTNAME),
    ("Active Identity",    ACTIVE_IDENTITY),
    ("Service Account",    f"{SERVICE_ACCOUNT} ({SA_STATUS}, <GoogleAccessToken> ready)"),
    ("Gemini Readiness",   f"PASS ({GEMINI_MODEL} @ {VERTEX_LOCATION})"),
    ("AFT Toolkit",        f"READY ({aft_ver})"),
    ("Local Repo (AFT)",   f"{REPO_DIR} (branch={REPO_BRANCH}, {num_templates} templates, {num_features} features)"),
]:
    print(f"{label:<20}: {val}")
print("=" * 72)
print("PREFLIGHT PASSED — safe to proceed")
